In [3]:
import re
import duckdb
import pandas as pd

# 1. Sample raw job description text (simulating a scraped page)
sample_job = """
    We are looking for a Junior Python Developer based in Kuching.
    Requirements:
    - 1+ years of experience with Python, FastAPI, and SQL.
    - Familiarity with Docker and AWS/Cloud certifications is a huge plus.
    - Salary: RM 3,500 - RM 5,000 per month.
"""

# 2. Define extraction logic using regex
def extract_job_details(text):
    # Normalize text
    text_lower = text.lower()
    
    # Define keywords to search for in tech stack
    target_skills = ['python', 'docker', 'aws', 'cloud', 'fastapi', 'sql', 'react', 'laravel']
    found_skills = [skill.upper() for skill in target_skills if re.search(r'\\b' + skill + r'\\b', text_lower)]
    
    # Extract salary pattern (e.g., RM X,XXX - RM Y,YYY)
    salary_match = re.search(r'rm\\s*[\\d,]+\\s*-\\s*rm\\s*[\\d,]+', text_lower)
    salary_range = salary_match.group(0).upper() if salary_match else "Not Specified"
    
    return {
        "skills": found_skills,
        "salary": salary_range
    }

# Run extraction
parsed_data = extract_job_details(sample_job)
print("Extracted Data:", parsed_data)

# 3. Test local DuckDB integration
# Create/connect to local DuckDB file in your data folder
con = duckdb.connect(database='../data/job_market.duckdb', read_only=False)

# Create table if not exists
con.execute("""
    CREATE TABLE IF NOT EXISTS job_listings (
        id INTEGER,
        title VARCHAR,
        skills VARCHAR[],
        salary_range VARCHAR
    )
""")

# Insert the parsed sample data
con.execute("""
    INSERT INTO job_listings VALUES (1, 'Junior Python Developer', ?, ?)
""", [parsed_data['skills'], parsed_data['salary']])

# Query back to verify
df_result = con.execute("SELECT * FROM job_listings").fetchdf()
print("\\nDuckDB Query Result:")
display(df_result)

con.close()

Extracted Data: {'skills': [], 'salary': 'Not Specified'}
\nDuckDB Query Result:


,id,title,skills,salary_range
0,1,Junior Python Developer,[],Not Specified
